# CTF - Alternative stub column parser

This notebook is to develop a stub column parser that works similarly to the crown parser. In other words, a parser that builds a tree by scanning columns left to right across the stub columns.

## Data source urls

Workbooks of immediate interest are in the [statistics on registered foreign residents that are provided by the Immigration Services Agency](https://www.moj.go.jp/isa/policies/statistics/toukei_ichiran_touroku.html). 

In [1]:
url12120101 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315227&fileKind=0"
url12120102 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315228&fileKind=0"
url12120201 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315229&fileKind=0"
url12120202 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315230&fileKind=0"
url121203 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315231&fileKind=0"
url121204 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315232&fileKind=0"
url12120500 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315233&fileKind=0"
url12120501 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315234&fileKind=0"
url12120502 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315235&fileKind=0"
url12120503 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315236&fileKind=0"
url12120504 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315237&fileKind=0"
url12120505 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315238&fileKind=0"
url12120600 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315239&fileKind=0"
url12120601 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315240&fileKind=0"
url12120602 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315241&fileKind=0"
url12120603 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315242&fileKind=0"
url12120604 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315303&fileKind=0"
url12120605 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000021315304&fileKind=0"
url121207 = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000022401560&fileKind=0"


## Import the module and download a file

In [2]:
from estatjp.xlsx import xlsx
from pathlib import Path

theurl = url121207
dataset = "201212"
ROOT_DIR = Path.cwd().parent.parent
dataraw = ROOT_DIR / Path(".data/raw/isa") / dataset
if Path.exists(dataraw) == False:
    Path.mkdir(dataraw)
dataprocessed = ROOT_DIR / Path(".data/processed/isa") / dataset
if Path.exists(dataprocessed) == False:
    Path.mkdir(dataprocessed)
newdata = xlsx.download_to_folder(theurl, dataraw, overwrite=True, verbose=True)
print('source: ' + str(newdata))
if newdata != None:
    processeddata = dataprocessed / newdata.name
    processeddata = processeddata.with_suffix(".csv")
    print('target: ' + str(processeddata))

{'Content-Type': 'application/octet-stream', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Strict-Transport-Security': 'max-age=0', 'X-Zen-Fury': '2b6dc629185fea1c0bffc0c087f4d82efe98bfa7', 'Date': 'Sat, 14 Nov 2026 05:06:11 GMT', 'Server': 'ZENEDGE', 'X-Content-Type-Options': 'nosniff', 'X-Cache-Status': 'NOTCACHED', 'X-Cache': 'MISS', 'Content-Encoding': 'gzip', 'Cache-Control': 'private', 'X-XSS-Protection': '1; mode=block', 'Content-Disposition': "attachment; filename*=UTF-8''12-12-07.xlsx", 'Accept-Ranges': 'bytes', 'X-Frame-Options': 'SAMEORIGIN', 'X-Cdn': 'Served-By-Zenedge'}
source: c:\Users\Alan\Documents\GitHub\estatjp\.data\raw\isa\201212\12-12-07.xlsx
target: c:\Users\Alan\Documents\GitHub\estatjp\.data\processed\isa\201212\12-12-07.csv


## Examine the worksheets and look for pivot tables or column-orientation

There are four known dataformats among the xlsx files
from E-Stat: cross tabulated format (CTF), column-oriented format
(COF), pivot (PVT), and pivot but no cache records. 
Determine format from file.

In [3]:
from estatjp.xlsx.xlsx import get_dataformat

dataformat = get_dataformat(newdata, verbose=True)
print('data format: ' + dataformat)

['12-12-07']
title: 12-12-07
dimensions: A2:M2302
print_title_rows: $1:$4
_pivots len: 0
merged_cells len: 1178
tableno: 07
data format: CTF


## Extract title information

Use `estatjp.xlsx.ctf.parse_title_information()`

In [4]:
from openpyxl import load_workbook
from estatjp.xlsx.ctf import parse_title_information

wb = load_workbook(newdata, data_only=True)
ws = wb.active
titleresults = parse_title_information(ws)
print(titleresults)

{'table_name': '12-12-07', 'time_code': 2012001212, 'stats_date': '2012年12月', 'title_population': '在留外国人', 'sub_population': '<NA>', 'cat01_label': '国籍・地域', 'cat01_toplabel': '地域', 'cat01_sublabel': '国籍', 'cat02_label': '市区町村', 'cat02_toplabel': '市区町村', 'cat02_sublabel': '<NA>'}


In [ ]:
from openpyxl import load_workbook
from openpyxl.worksheet.cell_range import (CellRange)
from openpyxl.cell.cell import Cell
from openpyxl.cell.cell import MergedCell

verbose = True
wsheet = ws
ranges_result = {
    'valrange' : None,
    'numrange' : None,
}
valcellrange = None
numcellrange = None

coord = wsheet.dimensions
stated_range = CellRange(coord)
found_max_col = stated_range.min_col
startrow = stated_range.min_row
if wsheet._print_rows != None:
    startrow = wsheet._print_rows.min_row
rowiter = wsheet.iter_rows(
                    min_row=startrow, 
                    max_row=stated_range.max_row, 
                    min_col=stated_range.min_col, 
                    max_col=stated_range.max_col)
for row in rowiter:
    hitcount = 0
    hitcellrange = None
    blankcellrange = None
    blankcount = 0
    max_val_col = 0
    ncellrange = None
    ncellcount = 0
    for idxcell, cell in enumerate(row):
        if (isinstance(cell, Cell) and cell.value != None):
            hitcount += 1
            thiscellrange = CellRange(cell.coordinate)
            if isinstance(hitcellrange, CellRange):
                hitcellrange = hitcellrange.union(thiscellrange)
            elif isinstance(cell, MergedCell):
                hitcellrange = hitcellrange.union(thiscellrange)
            else:
                hitcellrange = thiscellrange
            if idxcell+1 > max_val_col:
                max_val_col = cell.column
                if max_val_col > found_max_col:
                    found_max_col = max_val_col
            if cell.data_type == 'n':
                ncellcount += 1
                if isinstance(ncellrange, CellRange):
                    ncellrange = ncellrange.union(thiscellrange)
                else:
                    ncellrange = thiscellrange
        elif (isinstance(cell, Cell) and cell.value == None):
            blankcount += 1
            thiscellrange = CellRange(cell.coordinate)
            if isinstance(blankcellrange, CellRange):
                blankcellrange = blankcellrange.union(thiscellrange)
            else:
                blankcellrange = thiscellrange
#    if isinstance(numcellrange, CellRange) == True and blankcount > hitcount and max_val_col < found_max_col:
#        break
    if isinstance(valcellrange, CellRange) and isinstance(hitcellrange, CellRange):
        valcellrange = valcellrange.union(hitcellrange)
    elif isinstance(hitcellrange, CellRange) and hitcount > blankcount:
        valcellrange = hitcellrange
    if isinstance(numcellrange, CellRange) and isinstance(ncellrange, CellRange):
        numcellrange = numcellrange.union(ncellrange)
    elif isinstance(hitcellrange, CellRange) and hitcount > blankcount:
        numcellrange = ncellrange
    if verbose == True:
        print(str(hitcount) + ' values out of ' + str(len(row)) + ' max column=' + str(max_val_col) + ' blanks: ' + str(blankcount))
        print(str(ncellcount) + ' numerical values out of ' + str(len(row)) + ' max column=' + str(max_val_col))
if verbose == True:
    print('Value cell range: ' + str(valcellrange))
    print('Numerical cell range: ' + str(numcellrange))

print('Value cell range bottom: ' + str(valcellrange.bottom[0][0]))
print('Numeric cell range bottom: ' + str(numcellrange.bottom[0][0]))

ranges_result.update({'valrange': valcellrange})
ranges_result.update({'numrange': numcellrange})

if isinstance(numcellrange, CellRange) == True:
    if valcellrange.bottom[0][0] > numcellrange.bottom[0][0]:
        overrun = valcellrange.bottom[0][0] - numcellrange.bottom[0][0]
        valcellrange.shrink(bottom=overrun)
    ncrsize = numcellrange.size
    if verbose == True:
        print(ncrsize)
    colheaderrange = valcellrange.__copy__()
    colheaderrange.shrink(bottom=ncrsize['rows'])
    if verbose == True:
        print('Column header range: ' + str(colheaderrange))

    rowindexrange = valcellrange.__copy__()
    rowindexrange.shrink(right=ncrsize['columns'])
    if verbose == True:
        print('Stub column range: ' + str(rowindexrange))
    stubcellrange = colheaderrange.intersection(rowindexrange)
    if verbose == True:
        print('Stub cell range: ' + str(stubcellrange))
    scrsize = stubcellrange.size
    rowindexrange.shrink(top=scrsize['rows'])
    colcount = rowindexrange.size['columns']
    if verbose == True:
        print('Stub row range: ' + str(rowindexrange))
    depvarheadrange = colheaderrange.__copy__()
    depvarheadrange.shrink(left=scrsize['columns'])
    if verbose == True:
        print('Dependent column header range: ' + str(depvarheadrange))

ranges_result.update({'colheadrange': colheaderrange})
ranges_result.update({'depvarheadrange': depvarheadrange})
ranges_result.update({'stubcellrange': stubcellrange})
ranges_result.update({'stubcolrange': rowindexrange})

print(ranges_result)

0 values out of 13 max column=0 blanks: 13
0 numerical values out of 13 max column=0
1 values out of 13 max column=1 blanks: 0
0 numerical values out of 13 max column=1
0 values out of 13 max column=0 blanks: 13
0 numerical values out of 13 max column=0
11 values out of 13 max column=13 blanks: 0
0 numerical values out of 13 max column=13
12 values out of 13 max column=13 blanks: 0
10 numerical values out of 13 max column=13
11 values out of 13 max column=13 blanks: 2
10 numerical values out of 13 max column=13
11 values out of 13 max column=13 blanks: 2
10 numerical values out of 13 max column=13
11 values out of 13 max column=13 blanks: 2
10 numerical values out of 13 max column=13
11 values out of 13 max column=13 blanks: 2
10 numerical values out of 13 max column=13
11 values out of 13 max column=13 blanks: 2
10 numerical values out of 13 max column=13
11 values out of 13 max column=13 blanks: 2
10 numerical values out of 13 max column=13
11 values out of 13 max column=13 blanks: 2

## Build the stub column tree

Use package [`treelib`](https://treelib.readthedocs.io/en/latest/modules.html).

This borrows from the `treecrown` code in [non-pivot-xlsx-03](non-pivot-xlsx-03.ipynb).

In [36]:
import re
from openpyxl import Workbook
from openpyxl.cell.cell import Cell
from openpyxl.cell.cell import MergedCell
from openpyxl.worksheet.cell_range import (MultiCellRange, CellRange)
from openpyxl.styles.alignment import Alignment
from treelib import Tree, Node

stubtree = Tree()
# put title derived parameters in the root
stubtree.create_node(titleresults['table_name'], 'root',data={'dfdict':{}})
# create children for stub column (cat01) and dependent columns (cat02)
stubtree.create_node(titleresults["cat01_label"], 'cat01', parent='root', 
                 data={'value': '<NV>', 'dfdict':{
                                'table_name': titleresults['table_name'], 
                                'time_code': titleresults['time_code'],
                                'stats_date': titleresults['stats_date'],
                                'title_population': titleresults["title_population"],
                                'sub_population': titleresults["sub_population"],
                                'cat01': titleresults["cat01_label"] }})
topdepth = max(stubtree.level(node.identifier)
              for node in stubtree.all_nodes())
stubcolumnrange = ranges_result['stubcolrange']
previouscol = [None] * stubcolumnrange.size['rows']
topprnt = 'cat01'
prnt = topprnt
for idxcol, stubcol in enumerate(ws.iter_cols(
                                        min_col=stubcolumnrange.min_col, 
                                        min_row=stubcolumnrange.min_row, 
                                        max_col=stubcolumnrange.max_col, 
                                        max_row=stubcolumnrange.max_row)):
    prevvalcell = None
    for idxcell, cell in enumerate(stubcol):
        hasValue = cell.value != None # mimics merge by checking for no value
        # this second is for cells merged by selecting "Merge cells"
        inMergedCells = ws.merged_cells.__contains__(cell.coordinate)
        isMergeHead = hasValue == True and inMergedCells == True
        isMergedCell = hasValue == False and inMergedCells == True

        if previouscol[idxcell] != None:
            prnt = previouscol[idxcell]

        dfdictentry = {}
        val = '<NV>'    
        if cell.value == None: # 
            val = '<NV>'
            if len(stubtree.children(prnt)) == 0:
                stubtree.create_node(cell.coordinate + ' ' + val, cell.coordinate, parent=prnt, 
                                data={'MergeType': 'Cell', 'datatype': 'Cell', 'value': val, 'dfdict': dfdictentry})
                prevvalcell = None
                previouscol[idxcell] = cell.coordinate
            else:
                previouscol[idxcell] = prevvalcell
        else:
            val = re.sub(r'[\s\n\u3000]' , '', cell.value)

            if isMergeHead:
                stubtree.create_node(cell.coordinate + ' ' + val, cell.coordinate, parent=prnt, 
                                data={'MergeType': 'MergeHead', 'datatype': 'Cell', 'value': val, 'dfdict': dfdictentry})
                prevvalcell = cell.coordinate
                previouscol[idxcell] = cell.coordinate
            elif isinstance(cell, Cell):
                stubtree.create_node(cell.coordinate + ' ' + val, cell.coordinate, parent=prnt, 
                                data={'MergeType': 'Cell', 'datatype': 'Cell', 'value': val, 'dfdict': dfdictentry})
                prevvalcell = cell.coordinate
                previouscol[idxcell] = cell.coordinate
#        if isMergedCell:
#            stubtree.create_node(cell.coordinate + ' ' + val, cell.coordinate, parent=prnt, 
#                            data={'MergeType': 'MergedCell', 'datatype': 'MergedCell', 'value': val, 'dfdict': dfdictentry})
#            previouscol[idxcell] = cell.coordinate

stubtree.show(sorting=False) 


12-12-07
└── 国籍・地域
    ├── A5 北海道
    │   ├── B5 札幌市
    │   │   ├── C5 <NV>
    │   │   ├── C6 中央区
    │   │   ├── C7 北区
    │   │   ├── C8 東区
    │   │   ├── C9 白石区
    │   │   ├── C10 豊平区
    │   │   ├── C11 南区
    │   │   ├── C12 西区
    │   │   ├── C13 厚別区
    │   │   ├── C14 手稲区
    │   │   └── C15 清田区
    │   ├── B16 函館市
    │   │   └── C16 <NV>
    │   ├── B17 小樽市
    │   │   └── C17 <NV>
    │   ├── B18 旭川市
    │   │   └── C18 <NV>
    │   ├── B19 室蘭市
    │   │   └── C19 <NV>
    │   ├── B20 釧路市
    │   │   └── C20 <NV>
    │   ├── B21 帯広市
    │   │   └── C21 <NV>
    │   ├── B22 北見市
    │   │   └── C22 <NV>
    │   ├── B23 夕張市
    │   │   └── C23 <NV>
    │   ├── B24 岩見沢市
    │   │   └── C24 <NV>
    │   ├── B25 網走市
    │   │   └── C25 <NV>
    │   ├── B26 留萌市
    │   │   └── C26 <NV>
    │   ├── B27 苫小牧市
    │   │   └── C27 <NV>
    │   ├── B28 稚内市
    │   │   └── C28 <NV>
    │   ├── B29 美唄市
    │   │   └── C29 <NV>
    │   ├── B30 芦別市
    │   │   └── C30 <NV>
    │   ├── B3

In [ ]:
from estatjp.xlsx.ctf import paths_to_leaves_data

stubleaves = paths_to_leaves_data(stubtree, field='dfdict',makeuniform=True, noneform='<NA>')
stubleaves

In [37]:
# Now put the rest of dfdict entries in the nodes
max_depth = max(stubtree.level(node.identifier)
            for node in stubtree.all_nodes())
labeldict = [
    {'cat01lvl0label': '階級0'},
    {'cat01lvl1label': '階級1'},
    {'cat01lvl2label': '階級2'},
    {'cat01lvl3label': '階級3'},
    {'cat01lvl4label': '階級4'}
]
for i in range(topdepth+1,max_depth+1):
    level_nodes = [node for node in stubtree.all_nodes() if stubtree.level(node.identifier) == i]
    for node in level_nodes:
        thisdata = node.data.copy()
        thisdfdict = thisdata['dfdict'].copy()
        label = labeldict[i-topdepth]
        thisdfdict.update(label)
        thisdfdict.update({'cat01lvl' + str(i-topdepth): thisdata['value']})
        node.data.update({'dfdict': thisdfdict})


In [38]:
from estatjp.xlsx.ctf import paths_to_leaves_data

stubleaves = paths_to_leaves_data(stubtree, field='dfdict',makeuniform=True, noneform='<NA>')
stubleaves

{'C5': {'table_name': '12-12-07',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl1label': '階級1',
  'cat01lvl1': '北海道',
  'cat01lvl2label': '階級2',
  'cat01lvl2': '札幌市',
  'cat01lvl3label': '階級3',
  'cat01lvl3': '<NV>'},
 'C6': {'table_name': '12-12-07',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl1label': '階級1',
  'cat01lvl1': '北海道',
  'cat01lvl2label': '階級2',
  'cat01lvl2': '札幌市',
  'cat01lvl3label': '階級3',
  'cat01lvl3': '中央区'},
 'C7': {'table_name': '12-12-07',
  'time_code': 2012001212,
  'stats_date': '2012年12月',
  'title_population': '在留外国人',
  'sub_population': '<NA>',
  'cat01': '国籍・地域',
  'cat01lvl1label': '階級1',
  'cat01lvl1': '北海道',
  'cat01lvl2label': '階級2',
  'cat01lvl2': '札幌市',
  'cat01lvl3label': '階級3',
  'cat01lvl3': '北区'},
 'C8': {'table_name': '12-12-07',
  'time_code': 201